In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

In [2]:
dataset_path = "../torgo"

In [3]:
print("Dataset path:", dataset_path)
print(os.listdir(dataset_path))

Dataset path: ../torgo
['F_Con', 'F_Dys', 'M_Con', 'M_Dys']


In [4]:
group_information={
    "F_Con": {
        "gender":"Female",
        "condition":"Control"
    },
    "F_Dys": {
        "gender":"Female",
        "condition":"Dysarthria"
    },
    "M_Con": {
        "gender":"Male",
        "condition":"Control"
    },
    "M_Dys": {
        "gender":"Male",
        "condition":"Dysarthria"
    }
}
for group in group_information.keys():
    print(group,"->", "Gender:", group_information[group]["gender"], "," ,"Condition:", group_information[group]["condition"])



F_Con -> Gender: Female , Condition: Control
F_Dys -> Gender: Female , Condition: Dysarthria
M_Con -> Gender: Male , Condition: Control
M_Dys -> Gender: Male , Condition: Dysarthria


In [8]:
records = []

for group in group_information:

    group_path = os.path.join(dataset_path, group)

    for root, dirs, files in os.walk(group_path):

        for file in files:

            if file.lower().endswith(".wav"):

                file_path = os.path.join(root, file)

                records.append({
                    "file": file_path,
                    "group": group,
                    "gender": group_information[group]["gender"],
                    "condition": group_information[group]["condition"]
                })

In [9]:
df = pd.DataFrame(records)
df.head()

,file,group,gender,condition
0,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control
1,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control
2,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control
3,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control
4,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control


In [10]:
print("Number of rows:", len(df))
print("Shape:", df.shape)

Number of rows: 17635
Shape: (17635, 4)


In [11]:
df["condition"].value_counts()

condition
Control       11456
Dysarthria     6179
Name: count, dtype: int64

In [12]:
df["gender"].value_counts()

gender
Male      10566
Female     7069
Name: count, dtype: int64

In [13]:
pd.crosstab(
    df["gender"],
    df["condition"]
)

condition,Control,Dysarthria
gender,,
Female,4678,2391
Male,6778,3788


In [28]:
def extract_metadata(filename):

    parts = filename.split("_")

    microphone = parts[1]

    speaker_session = parts[2]

    recording_number = parts[3].replace(".wav", "")

    # Check whether a session is present
    if "S" in speaker_session:

        s_index = speaker_session.find("S")

        speaker = speaker_session[:s_index]
        session = speaker_session[s_index:]

    else:

        speaker = speaker_session
        session = None

    return microphone, speaker, session, recording_number

In [29]:
df["filename"] = df["file"].apply(os.path.basename)
print(df[["file", "filename"]].head())

                                                file  \
0  ../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...   
1  ../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...   
2  ../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...   
3  ../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...   
4  ../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...   

                        filename  
0  wav_arrayMic_FC01S01_0001.wav  
1  wav_arrayMic_FC01S01_0002.wav  
2  wav_arrayMic_FC01S01_0003.wav  
3  wav_arrayMic_FC01S01_0004.wav  
4  wav_arrayMic_FC01S01_0005.wav  


In [30]:
df[
    ["microphone", "speaker", "session", "recording_number"]
] = df["filename"].apply(
    lambda x: pd.Series(extract_metadata(x))
)

In [31]:
bad_files = df[
    df["filename"].apply(lambda x: len(x.split("_")) < 4)
]

print(bad_files[["file", "filename"]].to_string())

Empty DataFrame
Columns: [file, filename]
Index: []


In [32]:
df.head(10)

,file,group,gender,condition,filename,microphone,speaker,session,recording_number
0,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0001.wav,arrayMic,FC01,S01,0001
1,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0002.wav,arrayMic,FC01,S01,0002
2,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0003.wav,arrayMic,FC01,S01,0003
3,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0004.wav,arrayMic,FC01,S01,0004
4,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0005.wav,arrayMic,FC01,S01,0005
5,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0006.wav,arrayMic,FC01,S01,0006
6,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0007.wav,arrayMic,FC01,S01,0007
7,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0008.wav,arrayMic,FC01,S01,0008
8,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0009.wav,arrayMic,FC01,S01,0009
9,../torgo\F_Con\wav_arrayMic_FC01S01\wav_arrayM...,F_Con,Female,Control,wav_arrayMic_FC01S01_0010.wav,arrayMic,FC01,S01,0010


In [33]:
print("Number of unique speakers:", df["speaker"].nunique())

Number of unique speakers: 15


In [34]:
print(df["microphone"].value_counts())

microphone
arrayMic    9417
headMic     8218
Name: count, dtype: int64


In [35]:
print(df["group"].value_counts())

group
M_Con    6778
F_Con    4678
M_Dys    3788
F_Dys    2391
Name: count, dtype: int64


In [39]:
print(df["session"].isna().sum())

268


In [40]:
print(
    df[df["session"].isna()][
        ["filename", "group", "speaker", "microphone"]
    ].head(20).to_string(index=False)
)

                 filename group speaker microphone
wav_arrayMic_F01_0001.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0002.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0003.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0004.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0005.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0006.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0007.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0008.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0009.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0010.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0011.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0012.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0013.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0014.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0015.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0016.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0017.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0018.wav F_Dys     F01   arrayMic
wav_arrayMic_F01_0019.wav F_Dys

In [41]:
print(extract_metadata("wav_arrayMic_FC01S01_0001.wav"))

print(extract_metadata("wav_arrayMic_F01_0001.wav"))

print(extract_metadata("wav_headMic_M05S02_0009.wav"))

('arrayMic', 'FC01', 'S01', '0001')
('arrayMic', 'F01', None, '0001')
('headMic', 'M05', 'S02', '0009')


In [42]:
print(df["speaker"].value_counts())

print("\nUnique speakers:", df["speaker"].nunique())

print("\nMissing sessions:", df["session"].isna().sum())

speaker
FC02    2261
MC01    2216
FC03    1997
MC03    1728
MC04    1675
F03     1425
MC02    1159
M03      837
M02      809
M01      772
M04      719
F04      698
M05      651
FC01     420
F01      268
Name: count, dtype: int64

Unique speakers: 15

Missing sessions: 268


In [44]:
df.to_csv("../torgo_metadata.csv", index=False)

print("Metadata saved successfully!")

Metadata saved successfully!
